# Phase 9A AUC-Oriented Ranking Diagnostics

This notebook executes the authorized Phase 9A diagnostic pass for the Reto Tokio / GCI World NFL Draft Prediction project.

Scope:

- Read persisted Phase 8 OOF predictions only.
- Recompute and audit OOF metrics.
- Produce ranking, slice, score-distribution and disagreement diagnostics.
- Do not train models, tune hyperparameters, fit calibration, create submissions, use leaderboard feedback, or open Phase 10/11.

In [1]:
from __future__ import annotations

import csv
import hashlib
import math
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score


PROJECT_SEED = 42
EXPERIMENT_ID = "phase9a_auc_ranking_diagnostics_v1"
AUTHORIZED_HEAD = "020743681e10b94e26631cd4db01d244b45ba0e8"
REPO_ROOT = Path(r"C:\GitHub\reto_Tokio")
RARE_SCHOOL_THRESHOLD = 5
FOLD_SHA16_EXPECTED = "96937649526bcadb"
POSITIVE_LABEL = 1

assert (REPO_ROOT / ".git").exists(), f"Not a repo root: {REPO_ROOT}"
os.chdir(REPO_ROOT)


def run_cmd(args: list[str]) -> str:
    result = subprocess.run(args, cwd=REPO_ROOT, text=True, capture_output=True, check=True)
    return result.stdout.strip()


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def csv_rows(path: Path) -> int | None:
    try:
        return len(pd.read_csv(path))
    except Exception:
        return None


def safe_write_csv(df: pd.DataFrame, path: Path) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def safe_write_text(text: str, path: Path) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")


def fmt_float(value, ndigits: int = 12) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.{ndigits}f}"
    return str(value)


def markdown_table(rows: list[dict], columns: list[str], max_rows: int | None = None) -> str:
    if max_rows is not None:
        rows = rows[:max_rows]
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        vals = []
        for col in columns:
            val = row.get(col, "")
            if isinstance(val, (float, np.floating)):
                vals.append(fmt_float(val, 10))
            else:
                vals.append(str(val).replace("|", "\\|"))
        body.append("| " + " | ".join(vals) + " |")
    return "\n".join([header, sep] + body)


def auc_or_nan(y_true: pd.Series, y_score: pd.Series) -> float:
    if y_true.nunique(dropna=False) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


def ap_or_nan(y_true: pd.Series, y_score: pd.Series) -> float:
    if y_true.nunique(dropna=False) < 2:
        return float("nan")
    return float(average_precision_score(y_true, y_score))


def classify_verdict(model_key: str) -> str:
    if model_key == "m0_random_forest_frozen":
        return "anchor"
    if model_key == "m1_logistic_regression":
        return "carry"
    if model_key == "catboost":
        return "observe"
    return "drop-candidate"


def status_from_bool(ok: bool) -> str:
    return "PASS" if ok else "FAIL"


print("Phase 9A AUC/ranking diagnostics starting")
head = run_cmd(["git", "rev-parse", "HEAD"])
short_head = run_cmd(["git", "rev-parse", "--short", "HEAD"])
if head != AUTHORIZED_HEAD:
    raise RuntimeError(f"HEAD mismatch: expected {AUTHORIZED_HEAD}, observed {head}")

staged = run_cmd(["git", "diff", "--cached", "--name-only"])
if staged:
    raise RuntimeError(f"Staged files present before execution: {staged}")

forbidden = run_cmd([
    "git", "diff", "--name-only", "--",
    "data/input", "notebooks/_official", "references", "outputs/submissions",
    "logs/experiment_log.csv", ".vscode/settings.json",
])
if forbidden:
    raise RuntimeError(f"Forbidden-path diff before execution: {forbidden}")

diff_check = subprocess.run(["git", "diff", "--check"], cwd=REPO_ROOT, text=True, capture_output=True)
if diff_check.returncode != 0:
    raise RuntimeError(f"git diff --check failed before execution:\n{diff_check.stdout}\n{diff_check.stderr}")

main_log = REPO_ROOT / "logs" / "experiment_log.csv"
main_log_sha_before = sha256_file(main_log)

acceptance_path = REPO_ROOT / "docs" / "09_auc_ranking_diagnostics" / "phase9a_acceptance.md"
backlog_path = REPO_ROOT / "docs" / "09_auc_ranking_diagnostics" / "phase9a_improvement_backlog.md"
if acceptance_path.exists() or backlog_path.exists():
    raise RuntimeError("Phase 9A acceptance/backlog document already exists; Codex execution must not create or overwrite it.")

paths = {
    "oof_integrity": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_oof_integrity_report.csv",
    "auc_reproduction": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_auc_reproduction.csv",
    "global_metrics": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_global_metrics.csv",
    "topk_quantile": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_topk_quantile.csv",
    "fold_paired": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_fold_paired.csv",
    "slice_report": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_slice_report.csv",
    "score_distribution": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_score_distribution.csv",
    "disagreement": REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_disagreement.csv",
    "validation_report": REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_validation_report.md",
    "artifact_manifest": REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_artifact_manifest.csv",
    "candidate_log": REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_experiment_log_candidate.csv",
}
collisions = [str(p) for p in paths.values() if p.exists()]
if collisions:
    raise FileExistsError("Refusing to overwrite existing Phase 9A artifacts:\n" + "\n".join(collisions))

model_files = {
    "m0_random_forest_frozen": REPO_ROOT / "outputs" / "oof" / "phase8_model_family_comparison_v1_m0_random_forest_frozen_oof_predictions.csv",
    "m1_logistic_regression": REPO_ROOT / "outputs" / "oof" / "phase8_model_family_comparison_v1_m1_logistic_regression_oof_predictions.csv",
    "xgboost": REPO_ROOT / "outputs" / "oof" / "phase8_wave2_external_gbdt_v1_xgboost_oof_predictions.csv",
    "lightgbm": REPO_ROOT / "outputs" / "oof" / "phase8_wave2_external_gbdt_v1_lightgbm_oof_predictions.csv",
    "catboost": REPO_ROOT / "outputs" / "oof" / "phase8_wave2_external_gbdt_v1_catboost_oof_predictions.csv",
}
expected_auc = {
    "m0_random_forest_frozen": 0.8116502602456482,
    "m1_logistic_regression": 0.8270821069632867,
    "xgboost": 0.8113477083751576,
    "lightgbm": 0.8062204891415921,
    "catboost": 0.8202943968641223,
}
source_status = {
    "m0_random_forest_frozen": "reference_anchor",
    "m1_logistic_regression": "candidate_with_warning",
    "xgboost": "no_qualifying_evidence",
    "lightgbm": "no_qualifying_evidence",
    "catboost": "escalated_candidate_with_warning",
}
display_name = {
    "m0_random_forest_frozen": "M0 RandomForest frozen",
    "m1_logistic_regression": "M1 LogisticRegression",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM",
    "catboost": "CatBoost",
}

fold_path = REPO_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
if sha256_file(fold_path)[:16] != FOLD_SHA16_EXPECTED:
    raise RuntimeError("Frozen fold SHA mismatch")
folds = pd.read_csv(fold_path)
if len(folds) != 2781 or set(folds["fold"].unique()) != set(range(5)):
    raise RuntimeError("Frozen fold file failed row-count or label check")

train = pd.read_csv(REPO_ROOT / "data" / "input" / "train.csv")
if len(train) != 2781:
    raise RuntimeError(f"Unexpected train row count: {len(train)}")
if "Drafted" not in train.columns or "Id" not in train.columns:
    raise RuntimeError("Official train file missing Id or Drafted")

oofs: dict[str, pd.DataFrame] = {}
for model_key, path in model_files.items():
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    oofs[model_key] = df

anchor = oofs["m0_random_forest_frozen"].copy()
required_cols = ["Id", "fold", "y_true", "y_pred_proba"]
anchor_key = anchor[["Id", "fold", "y_true"]].copy()
anchor_fold_map = dict(zip(anchor["Id"], anchor["fold"]))
anchor_target_map = dict(zip(anchor["Id"], anchor["y_true"]))
fold_map = dict(zip(folds["Id"], folds["fold"]))

integrity_records = []
for model_key, df in oofs.items():
    schema_ok = list(df.columns) == required_cols
    row_count_ok = len(df) == 2781
    duplicate_id_fold_count = int(df.duplicated(["Id", "fold"]).sum()) if {"Id", "fold"}.issubset(df.columns) else -1
    duplicate_id_count = int(df["Id"].duplicated().sum()) if "Id" in df.columns else -1
    finite_probs = bool(np.isfinite(df["y_pred_proba"]).all()) if "y_pred_proba" in df.columns else False
    probs_in_range = bool(df["y_pred_proba"].between(0, 1, inclusive="both").all()) if "y_pred_proba" in df.columns else False
    no_nan = bool(df[required_cols].notna().all().all()) if all(c in df.columns for c in required_cols) else False
    fold_labels_ok = set(df["fold"].unique()) == set(range(5)) if "fold" in df.columns else False
    fold_alignment_mismatches = int((df["Id"].map(fold_map) != df["fold"]).sum()) if {"Id", "fold"}.issubset(df.columns) else -1
    anchor_fold_mismatches = int((df["Id"].map(anchor_fold_map) != df["fold"]).sum()) if {"Id", "fold"}.issubset(df.columns) else -1
    anchor_target_mismatches = int((df["Id"].map(anchor_target_map) != df["y_true"]).sum()) if {"Id", "y_true"}.issubset(df.columns) else -1
    positive_count = int((df["y_true"] == 1).sum()) if "y_true" in df.columns else -1
    negative_count = int((df["y_true"] == 0).sum()) if "y_true" in df.columns else -1
    positive_rate = positive_count / len(df) if len(df) else float("nan")
    one_class_folds = []
    if {"fold", "y_true"}.issubset(df.columns):
        for fold, fold_df in df.groupby("fold"):
            if fold_df["y_true"].nunique() < 2:
                one_class_folds.append(str(fold))
    passed = all([
        schema_ok, row_count_ok, duplicate_id_fold_count == 0, duplicate_id_count == 0,
        finite_probs, probs_in_range, no_nan, fold_labels_ok,
        fold_alignment_mismatches == 0, anchor_fold_mismatches == 0,
        anchor_target_mismatches == 0, not one_class_folds,
    ])
    integrity_records.append({
        "experiment_id": EXPERIMENT_ID,
        "model_key": model_key,
        "path": str(model_files[model_key].relative_to(REPO_ROOT)),
        "schema_ok": schema_ok,
        "row_count": len(df),
        "row_count_ok": row_count_ok,
        "duplicate_id_fold_count": duplicate_id_fold_count,
        "duplicate_id_count": duplicate_id_count,
        "finite_probs": finite_probs,
        "probs_in_range": probs_in_range,
        "no_nan": no_nan,
        "fold_labels_ok": fold_labels_ok,
        "fold_alignment_mismatches": fold_alignment_mismatches,
        "anchor_fold_mismatches": anchor_fold_mismatches,
        "anchor_target_mismatches": anchor_target_mismatches,
        "one_class_folds": ";".join(one_class_folds),
        "positive_count": positive_count,
        "negative_count": negative_count,
        "positive_rate": positive_rate,
        "passed": passed,
    })

integrity_df = pd.DataFrame(integrity_records)
if not bool(integrity_df["passed"].all()):
    raise RuntimeError("OOF integrity gate failed:\n" + integrity_df.to_string())
safe_write_csv(integrity_df, paths["oof_integrity"])

auc_records = []
for model_key, df in oofs.items():
    recomputed = float(roc_auc_score(df["y_true"], df["y_pred_proba"]))
    accepted = expected_auc[model_key]
    abs_diff = abs(recomputed - accepted)
    auc_records.append({
        "experiment_id": EXPERIMENT_ID,
        "model_key": model_key,
        "accepted_auc": accepted,
        "recomputed_auc": recomputed,
        "abs_diff": abs_diff,
        "tolerance": 1e-9,
        "status": "PASS" if abs_diff <= 1e-9 else "FAIL",
    })
auc_df = pd.DataFrame(auc_records)
if not (auc_df["status"] == "PASS").all():
    raise RuntimeError("AUC reproduction gate failed:\n" + auc_df.to_string())
safe_write_csv(auc_df, paths["auc_reproduction"])

global_positive_rate = float(anchor["y_true"].mean())
global_negative_rate = 1.0 - global_positive_rate

global_records = []
fold_records = []
fold_auc_lookup = {}
fold_ap_lookup = {}
for model_key, df in oofs.items():
    y = df["y_true"].astype(int)
    p = df["y_pred_proba"].astype(float)
    roc_auc = float(roc_auc_score(y, p))
    avg_precision = float(average_precision_score(y, p))
    neg_avg_precision = float(average_precision_score(1 - y, 1 - p))
    brier = float(brier_score_loss(y, p))
    fold_aucs = []
    fold_aps = []
    for fold in sorted(df["fold"].unique()):
        fold_df = df[df["fold"] == fold]
        fold_auc = auc_or_nan(fold_df["y_true"], fold_df["y_pred_proba"])
        fold_ap = ap_or_nan(fold_df["y_true"], fold_df["y_pred_proba"])
        fold_brier = float(brier_score_loss(fold_df["y_true"], fold_df["y_pred_proba"]))
        fold_aucs.append(fold_auc)
        fold_aps.append(fold_ap)
        fold_auc_lookup[(model_key, int(fold))] = fold_auc
        fold_ap_lookup[(model_key, int(fold))] = fold_ap
        fold_records.append({
            "experiment_id": EXPERIMENT_ID,
            "model_key": model_key,
            "fold": int(fold),
            "n": int(len(fold_df)),
            "n_positive": int((fold_df["y_true"] == 1).sum()),
            "n_negative": int((fold_df["y_true"] == 0).sum()),
            "positive_rate": float(fold_df["y_true"].mean()),
            "roc_auc": fold_auc,
            "average_precision": fold_ap,
            "brier_score": fold_brier,
            "delta_auc_vs_m0": np.nan,
            "delta_auc_vs_m1": np.nan,
            "delta_ap_vs_m0": np.nan,
            "delta_ap_vs_m1": np.nan,
        })
    global_records.append({
        "experiment_id": EXPERIMENT_ID,
        "model_key": model_key,
        "display_name": display_name[model_key],
        "source_status": source_status[model_key],
        "technical_verdict": classify_verdict(model_key),
        "n": int(len(df)),
        "n_positive": int(y.sum()),
        "n_negative": int((1 - y).sum()),
        "positive_rate": global_positive_rate,
        "negative_rate": global_negative_rate,
        "roc_auc": roc_auc,
        "average_precision": avg_precision,
        "average_precision_baseline": global_positive_rate,
        "negative_class_average_precision": neg_avg_precision,
        "negative_class_average_precision_baseline": global_negative_rate,
        "brier_score": brier,
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs, ddof=0)),
        "fold_auc_scores": "|".join(f"{v:.12f}" for v in fold_aucs),
        "delta_auc_vs_m0": roc_auc - expected_auc["m0_random_forest_frozen"],
        "delta_auc_vs_m1": roc_auc - expected_auc["m1_logistic_regression"],
        "same_sign_positive_folds_vs_m0": np.nan,
        "same_sign_positive_folds_vs_m1": np.nan,
    })

fold_df = pd.DataFrame(fold_records)
for idx, row in fold_df.iterrows():
    fold = int(row["fold"])
    model_key = row["model_key"]
    fold_df.at[idx, "delta_auc_vs_m0"] = row["roc_auc"] - fold_auc_lookup[("m0_random_forest_frozen", fold)]
    fold_df.at[idx, "delta_auc_vs_m1"] = row["roc_auc"] - fold_auc_lookup[("m1_logistic_regression", fold)]
    fold_df.at[idx, "delta_ap_vs_m0"] = row["average_precision"] - fold_ap_lookup[("m0_random_forest_frozen", fold)]
    fold_df.at[idx, "delta_ap_vs_m1"] = row["average_precision"] - fold_ap_lookup[("m1_logistic_regression", fold)]

global_df = pd.DataFrame(global_records)
for idx, row in global_df.iterrows():
    model_key = row["model_key"]
    if model_key == "m0_random_forest_frozen":
        global_df.at[idx, "same_sign_positive_folds_vs_m0"] = 0
    else:
        deltas = [
            fold_auc_lookup[(model_key, fold)] - fold_auc_lookup[("m0_random_forest_frozen", fold)]
            for fold in range(5)
        ]
        global_df.at[idx, "same_sign_positive_folds_vs_m0"] = int(sum(delta > 0 for delta in deltas))
    if model_key == "m1_logistic_regression":
        global_df.at[idx, "same_sign_positive_folds_vs_m1"] = 0
    else:
        deltas = [
            fold_auc_lookup[(model_key, fold)] - fold_auc_lookup[("m1_logistic_regression", fold)]
            for fold in range(5)
        ]
        global_df.at[idx, "same_sign_positive_folds_vs_m1"] = int(sum(delta > 0 for delta in deltas))

safe_write_csv(global_df, paths["global_metrics"])
safe_write_csv(fold_df, paths["fold_paired"])

topk_records = []
top_k_values = [50, 100, 250, 500]
top_pct_values = [0.10, 0.20]
for model_key, df in oofs.items():
    sorted_df = df.sort_values(["y_pred_proba", "Id"], ascending=[False, True]).reset_index(drop=True)
    total_pos = int((sorted_df["y_true"] == 1).sum())
    total_neg = len(sorted_df) - total_pos
    for k in top_k_values:
        selected = sorted_df.head(k)
        pos_selected = int((selected["y_true"] == 1).sum())
        precision = pos_selected / k
        recall = pos_selected / total_pos
        lift = precision / global_positive_rate
        topk_records.append({
            "experiment_id": EXPERIMENT_ID,
            "model_key": model_key,
            "rank_region": "top_k_positive",
            "cut_label": f"top_{k}",
            "n_selected": k,
            "positives_selected": pos_selected,
            "negatives_selected": k - pos_selected,
            "precision": precision,
            "recall_capture": recall,
            "lift_vs_random": lift,
            "random_precision_baseline": global_positive_rate,
            "random_capture_baseline": k / len(sorted_df),
            "note": "Positive-class ranking head; diagnostic only.",
        })
        bottom = sorted_df.tail(k)
        neg_selected = int((bottom["y_true"] == 0).sum())
        neg_precision = neg_selected / k
        neg_recall = neg_selected / total_neg
        neg_lift = neg_precision / global_negative_rate
        topk_records.append({
            "experiment_id": EXPERIMENT_ID,
            "model_key": model_key,
            "rank_region": "bottom_k_negative",
            "cut_label": f"bottom_{k}",
            "n_selected": k,
            "positives_selected": k - neg_selected,
            "negatives_selected": neg_selected,
            "precision": neg_precision,
            "recall_capture": neg_recall,
            "lift_vs_random": neg_lift,
            "random_precision_baseline": global_negative_rate,
            "random_capture_baseline": k / len(sorted_df),
            "note": "Negative-class retrieval from low-score tail; diagnostic only.",
        })
    for pct in top_pct_values:
        k = int(round(len(sorted_df) * pct))
        selected = sorted_df.head(k)
        pos_selected = int((selected["y_true"] == 1).sum())
        precision = pos_selected / k
        recall = pos_selected / total_pos
        topk_records.append({
            "experiment_id": EXPERIMENT_ID,
            "model_key": model_key,
            "rank_region": "top_quantile_positive",
            "cut_label": f"top_{int(pct * 100)}pct",
            "n_selected": k,
            "positives_selected": pos_selected,
            "negatives_selected": k - pos_selected,
            "precision": precision,
            "recall_capture": recall,
            "lift_vs_random": precision / global_positive_rate,
            "random_precision_baseline": global_positive_rate,
            "random_capture_baseline": k / len(sorted_df),
            "note": "Positive-class quantile enrichment; diagnostic only.",
        })

topk_df = pd.DataFrame(topk_records)
safe_write_csv(topk_df, paths["topk_quantile"])

physical_test_columns = [
    "Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps",
    "Broad_Jump", "Agility_3cone", "Shuttle",
]
diag = train[["Id", "Year", "Age", "Player_Type", "Position_Type", "Position", "School"] + physical_test_columns].copy()
diag["Age_missing"] = diag["Age"].isna().astype(int)
present_physical = [c for c in physical_test_columns if c in diag.columns]
diag["available_measurement_count"] = diag[present_physical].notna().sum(axis=1).astype(int)
diag["measurement_completeness_group"] = pd.cut(
    diag["available_measurement_count"],
    bins=[-1, 0, 3, 5, 6],
    labels=["none", "low", "partial", "complete"],
).astype(str)
school_counts = diag["School"].value_counts(dropna=False)
diag["frequent_vs_rare_school_group"] = diag["School"].map(school_counts).fillna(0).apply(
    lambda count: "frequent" if count >= RARE_SCHOOL_THRESHOLD else "rare"
)
diag["Year"] = diag["Year"].astype("Int64").astype(str)
diag["available_measurement_count"] = diag["available_measurement_count"].astype(str)
diag["Age_missing"] = diag["Age_missing"].astype(str)
for col in ["Player_Type", "Position_Type", "Position", "frequent_vs_rare_school_group"]:
    diag[col] = diag[col].fillna("missing").astype(str)

slice_specs = [
    ("Player_Type", 50, "established"),
    ("Position_Type", 50, "established"),
    ("Year", 50, "established"),
    ("Age_missing", 50, "established"),
    ("available_measurement_count", 50, "established"),
    ("measurement_completeness_group", 50, "established"),
    ("frequent_vs_rare_school_group", 50, "established_school_diagnostic_only"),
    ("Position", 100, "optional_fine_grained"),
]
slice_records = []
for slice_name, min_n, slice_role in slice_specs:
    values = sorted(diag[slice_name].dropna().unique(), key=lambda x: str(x))
    for value in values:
        ids = set(diag.loc[diag[slice_name] == value, "Id"])
        per_model = {}
        stats = {}
        for model_key, df in oofs.items():
            sub = df[df["Id"].isin(ids)].copy()
            n = int(len(sub))
            n_pos = int((sub["y_true"] == 1).sum()) if n else 0
            n_neg = int((sub["y_true"] == 0).sum()) if n else 0
            if n < min_n:
                status = "skipped_too_small"
                auc_val = np.nan
            elif n_pos == 0 or n_neg == 0:
                status = "skipped_one_class"
                auc_val = np.nan
            else:
                status = "computed"
                auc_val = float(roc_auc_score(sub["y_true"], sub["y_pred_proba"]))
            per_model[model_key] = auc_val
            stats[model_key] = (n, n_pos, n_neg, status)
        for model_key in model_files:
            n, n_pos, n_neg, status = stats[model_key]
            auc_val = per_model[model_key]
            m0_auc = per_model["m0_random_forest_frozen"]
            m1_auc = per_model["m1_logistic_regression"]
            delta_m0 = auc_val - m0_auc if status == "computed" and not pd.isna(m0_auc) else np.nan
            delta_m1 = auc_val - m1_auc if status == "computed" and not pd.isna(m1_auc) else np.nan
            guard_drop = bool(status == "computed" and n >= min_n and not pd.isna(delta_m0) and delta_m0 < -0.02)
            fragile = bool(n_pos < 20 and status == "computed")
            slice_records.append({
                "experiment_id": EXPERIMENT_ID,
                "model_key": model_key,
                "slice_name": slice_name,
                "slice_value": value,
                "slice_role": slice_role,
                "min_n_rule": min_n,
                "n": n,
                "n_positive": n_pos,
                "n_negative": n_neg,
                "positive_rate": n_pos / n if n else np.nan,
                "roc_auc": auc_val,
                "delta_auc_vs_m0": delta_m0,
                "delta_auc_vs_m1": delta_m1,
                "status": status,
                "warning_drop_gt_0_02_vs_m0": guard_drop,
                "fragile_positive_support_lt_20": fragile,
                "notes": "School diagnostic only; not a feature." if slice_name == "frequent_vs_rare_school_group" else "",
            })

slice_df = pd.DataFrame(slice_records)
safe_write_csv(slice_df, paths["slice_report"])

dist_records = []
quantile_points = [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
for model_key, df in oofs.items():
    for segment_name, segment_df in [("all", df), ("y_true_0", df[df["y_true"] == 0]), ("y_true_1", df[df["y_true"] == 1])]:
        probs = segment_df["y_pred_proba"].astype(float)
        q = probs.quantile(quantile_points).to_dict()
        dist_records.append({
            "experiment_id": EXPERIMENT_ID,
            "model_key": model_key,
            "record_type": "score_quantiles",
            "segment": segment_name,
            "n": int(len(segment_df)),
            "positive_rate": float(segment_df["y_true"].mean()) if len(segment_df) else np.nan,
            "mean_pred": float(probs.mean()) if len(probs) else np.nan,
            "std_pred": float(probs.std(ddof=0)) if len(probs) else np.nan,
            "min": float(q.get(0, np.nan)),
            "p01": float(q.get(0.01, np.nan)),
            "p05": float(q.get(0.05, np.nan)),
            "p10": float(q.get(0.10, np.nan)),
            "p25": float(q.get(0.25, np.nan)),
            "p50": float(q.get(0.50, np.nan)),
            "p75": float(q.get(0.75, np.nan)),
            "p90": float(q.get(0.90, np.nan)),
            "p95": float(q.get(0.95, np.nan)),
            "p99": float(q.get(0.99, np.nan)),
            "max": float(q.get(1.0, np.nan)),
            "brier_score": float(brier_score_loss(segment_df["y_true"], probs)) if segment_name == "all" else np.nan,
            "calibration_bin": "",
            "bin_lower": np.nan,
            "bin_upper": np.nan,
            "observed_rate": np.nan,
        })
    tmp = df.copy()
    tmp["calibration_bin"] = pd.qcut(tmp["y_pred_proba"].rank(method="first"), q=10, labels=False)
    for bin_id, bin_df in tmp.groupby("calibration_bin"):
        dist_records.append({
            "experiment_id": EXPERIMENT_ID,
            "model_key": model_key,
            "record_type": "calibration_decile_diagnostic_only",
            "segment": "all",
            "n": int(len(bin_df)),
            "positive_rate": float(bin_df["y_true"].mean()),
            "mean_pred": float(bin_df["y_pred_proba"].mean()),
            "std_pred": float(bin_df["y_pred_proba"].std(ddof=0)),
            "min": np.nan,
            "p01": np.nan,
            "p05": np.nan,
            "p10": np.nan,
            "p25": np.nan,
            "p50": np.nan,
            "p75": np.nan,
            "p90": np.nan,
            "p95": np.nan,
            "p99": np.nan,
            "max": np.nan,
            "brier_score": np.nan,
            "calibration_bin": int(bin_id),
            "bin_lower": float(bin_df["y_pred_proba"].min()),
            "bin_upper": float(bin_df["y_pred_proba"].max()),
            "observed_rate": float(bin_df["y_true"].mean()),
        })

dist_df = pd.DataFrame(dist_records)
safe_write_csv(dist_df, paths["score_distribution"])

wide = anchor[["Id", "fold", "y_true"]].copy()
for model_key, df in oofs.items():
    wide = wide.merge(df[["Id", "y_pred_proba"]].rename(columns={"y_pred_proba": model_key}), on="Id", how="left")

dis_records = []
pairs = [
    ("m1_logistic_regression", "catboost"),
    ("m1_logistic_regression", "m0_random_forest_frozen"),
    ("catboost", "m0_random_forest_frozen"),
    ("xgboost", "lightgbm"),
]
for a, b in pairs:
    spearman = float(wide[a].corr(wide[b], method="spearman"))
    kendall = float(wide[a].corr(wide[b], method="kendall"))
    pearson = float(wide[a].corr(wide[b], method="pearson"))
    for metric, value in [("spearman_rank_corr", spearman), ("kendall_rank_corr", kendall), ("pearson_score_corr", pearson)]:
        dis_records.append({
            "experiment_id": EXPERIMENT_ID,
            "record_type": "pair_correlation",
            "model_a": a,
            "model_b": b,
            "metric": metric,
            "value": value,
            "Id": "",
            "fold": "",
            "y_true": "",
            "proba_a": np.nan,
            "proba_b": np.nan,
            "rank_a": np.nan,
            "rank_b": np.nan,
            "rank_gap": np.nan,
            "direction": "",
            "note": "Diagnostic only; no blending or ensemble authorized.",
        })

rank_a = wide["m1_logistic_regression"].rank(ascending=False, method="average")
rank_b = wide["catboost"].rank(ascending=False, method="average")
tmp = wide.copy()
tmp["rank_m1"] = rank_a
tmp["rank_catboost"] = rank_b
tmp["m1_high_catboost_low_gap"] = tmp["rank_catboost"] - tmp["rank_m1"]
tmp["catboost_high_m1_low_gap"] = tmp["rank_m1"] - tmp["rank_catboost"]
for direction, gap_col in [
    ("m1_high_catboost_low", "m1_high_catboost_low_gap"),
    ("catboost_high_m1_low", "catboost_high_m1_low_gap"),
]:
    for _, row in tmp.sort_values(gap_col, ascending=False).head(25).iterrows():
        dis_records.append({
            "experiment_id": EXPERIMENT_ID,
            "record_type": "top_rank_divergence_case",
            "model_a": "m1_logistic_regression",
            "model_b": "catboost",
            "metric": "rank_gap",
            "value": float(row[gap_col]),
            "Id": int(row["Id"]),
            "fold": int(row["fold"]),
            "y_true": int(row["y_true"]),
            "proba_a": float(row["m1_logistic_regression"]),
            "proba_b": float(row["catboost"]),
            "rank_a": float(row["rank_m1"]),
            "rank_b": float(row["rank_catboost"]),
            "rank_gap": float(row[gap_col]),
            "direction": direction,
            "note": "Diagnostic case only; no manual relabeling or prediction editing.",
        })

dis_df = pd.DataFrame(dis_records)
safe_write_csv(dis_df, paths["disagreement"])

slice_warning_summary = (
    slice_df[(slice_df["status"] == "computed") & (slice_df["warning_drop_gt_0_02_vs_m0"])]
    .sort_values(["model_key", "delta_auc_vs_m0"])
)

verdict_notes = {
    "m0_random_forest_frozen": "Anchor/reference only; reproduced from accepted Phase 8 Wave 1 OOF.",
    "m1_logistic_regression": "Carry as candidate-with-warning; strongest global OOF but known Age_missing=1 slice fragility.",
    "catboost": "Observe as candidate-with-warning; useful global signal but escalated slice behavior from Phase 8 Wave 2 remains.",
    "xgboost": "Drop-candidate for now; Phase 8 Wave 2 no_qualifying_evidence preserved.",
    "lightgbm": "Drop-candidate for now; Phase 8 Wave 2 no_qualifying_evidence preserved.",
}
candidate_log_records = []
for _, row in global_df.iterrows():
    candidate_log_records.append({
        "experiment_id": EXPERIMENT_ID,
        "date": datetime.now(timezone.utc).date().isoformat(),
        "phase": "Phase 9A",
        "notebook_or_script": "notebooks/09a_auc_ranking_diagnostics.ipynb",
        "git_commit_or_status": f"{short_head}; dirty allowed only for untracked Phase 9A artifacts",
        "data_version": "official train rows; persisted OOF artifacts",
        "fold_strategy": f"frozen folds sha16={FOLD_SHA16_EXPECTED}",
        "random_seed": PROJECT_SEED,
        "feature_block": "accepted F2; diagnostics from persisted OOF only",
        "preprocessing_summary": "none fitted; read-only OOF diagnostics",
        "model_family": row["model_key"],
        "model_params_summary": "not trained in Phase 9A",
        "hpo_status": "not_run",
        "cv_auc_mean": row["roc_auc"],
        "cv_auc_std": row["fold_auc_std"],
        "fold_scores": row["fold_auc_scores"],
        "slice_metrics_available": True,
        "leakage_checks_passed": True,
        "submission_created": False,
        "submission_path": "",
        "public_lb_score_if_submitted": "",
        "notes": verdict_notes[row["model_key"]],
        "decision": row["technical_verdict"],
    })
candidate_log_df = pd.DataFrame(candidate_log_records)
safe_write_csv(candidate_log_df, paths["candidate_log"])

top_decile = topk_df[(topk_df["rank_region"] == "top_quantile_positive") & (topk_df["cut_label"] == "top_10pct")]
bottom_100 = topk_df[(topk_df["rank_region"] == "bottom_k_negative") & (topk_df["cut_label"] == "bottom_100")]
corr_summary = dis_df[dis_df["record_type"] == "pair_correlation"].copy()

backlog_seed = [
    {
        "ID": "H1",
        "Hypothesis": "Review M1 vs CatBoost disagreement cases before any future ensemble discussion.",
        "Evidence": "Phase 9A rank disagreement diagnostics.",
        "Future phase": "Phase 9B or later diagnostic; ensembles remain locked.",
    },
    {
        "ID": "H2",
        "Hypothesis": "Investigate Age_missing=1 separately due tiny positive support and repeated warning behavior.",
        "Evidence": "Age_missing=1 has n=435 and 8 positives; m1 and CatBoost warnings originate here in prior phases.",
        "Future phase": "Phase 9B diagnostic only.",
    },
    {
        "ID": "H3",
        "Hypothesis": "Use top-k and negative-tail retrieval as supporting evidence, not selection criteria.",
        "Evidence": f"Global positive rate is {global_positive_rate:.4f}; negatives are the minority side.",
        "Future phase": "Project-director strategic review.",
    },
    {
        "ID": "H4",
        "Hypothesis": "Defer all tuning/calibration/threshold/submission ideas until explicit Phase 10/11 authorization.",
        "Evidence": "Phase 9A is diagnostics only.",
        "Future phase": "Phase 10/11 locked.",
    },
]

report_lines = []
report_lines.append(f"# Phase 9A AUC/Raking Diagnostics Report")
report_lines.append("")
report_lines.append("## Executive Summary")
report_lines.append("")
report_lines.append("Phase 9A executed as a read-only diagnostic pass over persisted OOF predictions. No models were trained, no OOF predictions were edited, no HPO was run, no submissions were generated, and no final winner was selected.")
report_lines.append("")
report_lines.append(f"- Authorized HEAD: `{AUTHORIZED_HEAD}`")
report_lines.append(f"- Observed HEAD: `{head}`")
report_lines.append(f"- Experiment ID: `{EXPERIMENT_ID}`")
report_lines.append(f"- Global positive rate: `{global_positive_rate:.6f}` ({int(anchor['y_true'].sum())}/{len(anchor)})")
report_lines.append("- Main experiment log: read before and verified unchanged after artifact generation.")
report_lines.append("")
report_lines.append("## Integrity Gates")
report_lines.append("")
report_lines.append(markdown_table(integrity_df[["model_key", "row_count", "positive_count", "negative_count", "positive_rate", "passed"]].to_dict("records"), ["model_key", "row_count", "positive_count", "negative_count", "positive_rate", "passed"]))
report_lines.append("")
report_lines.append("## AUC Reproduction")
report_lines.append("")
report_lines.append(markdown_table(auc_df.to_dict("records"), ["model_key", "accepted_auc", "recomputed_auc", "abs_diff", "status"]))
report_lines.append("")
report_lines.append("## Global Metrics")
report_lines.append("")
report_lines.append(markdown_table(global_df.to_dict("records"), ["model_key", "source_status", "technical_verdict", "roc_auc", "average_precision", "negative_class_average_precision", "brier_score", "delta_auc_vs_m0", "delta_auc_vs_m1", "same_sign_positive_folds_vs_m0", "same_sign_positive_folds_vs_m1"]))
report_lines.append("")
report_lines.append("## Top-K And Quantile Diagnostics")
report_lines.append("")
report_lines.append("The positive class is the majority, so top-k and PR-style metrics are ranking-head diagnostics rather than rare-event retrieval substitutes for ROC-AUC. Random baselines are included in the CSV artifact.")
report_lines.append("")
report_lines.append(markdown_table(top_decile.to_dict("records"), ["model_key", "cut_label", "n_selected", "positives_selected", "precision", "recall_capture", "lift_vs_random"]))
report_lines.append("")
report_lines.append("Negative-tail retrieval is also reported because the negative class is the minority side.")
report_lines.append("")
report_lines.append(markdown_table(bottom_100.to_dict("records"), ["model_key", "cut_label", "n_selected", "negatives_selected", "precision", "recall_capture", "lift_vs_random"]))
report_lines.append("")
report_lines.append("## Fold-Level Paired Diagnostics")
report_lines.append("")
report_lines.append(markdown_table(fold_df.to_dict("records"), ["model_key", "fold", "roc_auc", "average_precision", "delta_auc_vs_m0", "delta_auc_vs_m1"], max_rows=30))
report_lines.append("")
report_lines.append("## Slice Diagnostics")
report_lines.append("")
if len(slice_warning_summary):
    report_lines.append("Mandatory slice drops greater than 0.02 AUC vs M0 are warnings for strategic review, not leakage findings.")
    report_lines.append("")
    report_lines.append(markdown_table(slice_warning_summary[["model_key", "slice_name", "slice_value", "n", "n_positive", "roc_auc", "delta_auc_vs_m0", "fragile_positive_support_lt_20"]].to_dict("records"), ["model_key", "slice_name", "slice_value", "n", "n_positive", "roc_auc", "delta_auc_vs_m0", "fragile_positive_support_lt_20"], max_rows=40))
else:
    report_lines.append("No computed mandatory slice degraded by more than 0.02 AUC vs M0.")
report_lines.append("")
report_lines.append("`School` was used only for the diagnostic frequent-vs-rare slice and never as a feature.")
report_lines.append("")
report_lines.append("## Score Distribution And Diagnostic Calibration")
report_lines.append("")
report_lines.append("Score quantiles, Brier score, and calibration deciles were computed descriptively. No calibration model was fitted.")
report_lines.append("")
report_lines.append(markdown_table(dist_df[(dist_df["record_type"] == "score_quantiles") & (dist_df["segment"] == "all")].to_dict("records"), ["model_key", "mean_pred", "std_pred", "p10", "p50", "p90", "brier_score"]))
report_lines.append("")
report_lines.append("## Disagreement Diagnostics")
report_lines.append("")
report_lines.append("Rank disagreement is diagnostic only. It does not authorize blending, stacking, ensembles, manual edits, or submissions.")
report_lines.append("")
report_lines.append(markdown_table(corr_summary.to_dict("records"), ["model_a", "model_b", "metric", "value"]))
report_lines.append("")
report_lines.append("## Technical Verdicts")
report_lines.append("")
report_lines.append("- `m0_random_forest_frozen`: anchor/reference.")
report_lines.append("- `m1_logistic_regression`: carry as candidate-with-warning for project-director review; no final winner selected.")
report_lines.append("- `catboost`: observe as candidate-with-warning; global signal exists but Phase 8 slice escalation remains material.")
report_lines.append("- `xgboost`: drop-candidate for now; no qualifying evidence preserved.")
report_lines.append("- `lightgbm`: drop-candidate for now; no qualifying evidence preserved.")
report_lines.append("")
report_lines.append("## Technical Backlog Seed")
report_lines.append("")
report_lines.append("This is a technical backlog seed only. Codex did not create `docs/09_auc_ranking_diagnostics/phase9a_improvement_backlog.md`; final strategic prioritization belongs to post-Codex review and the project director.")
report_lines.append("")
report_lines.append(markdown_table(backlog_seed, ["ID", "Hypothesis", "Evidence", "Future phase"]))
report_lines.append("")
report_lines.append("## Leakage And Scope Verdict")
report_lines.append("")
report_lines.append("Leakage verdict: pass for Phase 9A read-only diagnostics. No test fitting, no preprocessing fitting, no target encoding, no feature selection, no model training, no HPO, no leaderboard use, no external data, and no submissions occurred.")
report_lines.append("")
report_lines.append("## Explicit Non-Actions")
report_lines.append("")
report_lines.append("- No model was trained or retrained.")
report_lines.append("- No notebook outside `notebooks/09a_auc_ranking_diagnostics.ipynb` was created or executed.")
report_lines.append("- No HPO, ensemble, blending, stacking, calibration fitting, threshold tuning, or submission was run.")
report_lines.append("- `logs/experiment_log.csv` was not modified.")
report_lines.append("- Phase 10 and Phase 11 remain locked.")
report_lines.append("")
report_lines.append("Phase 9A diagnostics complete; this is technical evidence only. No final winner was selected and no submission was authorized. Strategic prioritization and acceptance remain for the project director. Phase 10 and Phase 11 remain locked.")

safe_write_text("\n".join(report_lines) + "\n", paths["validation_report"])

manifest_records = []
for key, path in paths.items():
    if key == "artifact_manifest":
        continue
    if not path.exists():
        raise RuntimeError(f"Expected artifact missing before manifest: {path}")
    manifest_records.append({
        "experiment_id": EXPERIMENT_ID,
        "artifact_key": key,
        "path": str(path.relative_to(REPO_ROOT)),
        "sha256": sha256_file(path),
        "row_count": csv_rows(path) if path.suffix.lower() == ".csv" else "",
        "created_utc": datetime.now(timezone.utc).isoformat(),
    })
manifest_records.append({
    "experiment_id": EXPERIMENT_ID,
    "artifact_key": "artifact_manifest",
    "path": str(paths["artifact_manifest"].relative_to(REPO_ROOT)),
    "sha256": "self_excluded_until_after_write",
    "row_count": "",
    "created_utc": datetime.now(timezone.utc).isoformat(),
})
manifest_df = pd.DataFrame(manifest_records)
safe_write_csv(manifest_df, paths["artifact_manifest"])

if sha256_file(main_log) != main_log_sha_before:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 9A execution")

post_forbidden = run_cmd([
    "git", "diff", "--name-only", "--",
    "data/input", "notebooks/_official", "references", "outputs/submissions",
    "logs/experiment_log.csv", ".vscode/settings.json",
])
if post_forbidden:
    raise RuntimeError(f"Forbidden-path diff after execution: {post_forbidden}")

print("Phase 9A diagnostics complete")
print(f"Artifacts written: {len(paths)}")
print(global_df[["model_key", "roc_auc", "average_precision", "negative_class_average_precision", "technical_verdict"]].to_string(index=False))

Phase 9A AUC/ranking diagnostics starting


Phase 9A diagnostics complete
Artifacts written: 11
              model_key  roc_auc  average_precision  negative_class_average_precision technical_verdict
m0_random_forest_frozen 0.811650           0.863811                          0.778719            anchor
 m1_logistic_regression 0.827082           0.874184                          0.790499             carry
                xgboost 0.811348           0.864025                          0.781012    drop-candidate
               lightgbm 0.806220           0.858134                          0.777041    drop-candidate
               catboost 0.820294           0.870463                          0.788618           observe
